# Laboratorio 2: Aprendizaje No Supervisado
## The Movie DB — Clustering, Reglas de Asociación, PCA y t-SNE
### Curso: Minería de Datos
---
**Dataset:** The Movie DB — 19,883 películas  
**Objetivo:** Aplicar técnicas de aprendizaje no supervisado para descubrir patrones
y segmentos en el mercado cinematográfico, generando recomendaciones estratégicas
para **CineVision Studios**.


---
## Sección 0: Configuración y Carga de Datos

Se instalan las librerías necesarias, se detecta automáticamente el archivo CSV
en el directorio de trabajo y se realiza una exploración inicial del dataset.


In [ ]:
# Instalar librerías requeridas (--quiet suprime la salida detallada de pip)
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn \
    mlxtend factor_analyzer pingouin scipy pyclustertend
print('Instalacion completada (o librerias ya presentes).')

### 0.1 Importación de Librerías


In [ ]:
import glob, os, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import scipy.cluster.hierarchy as sch

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from mlxtend.frequent_patterns import apriori, association_rules

from factor_analyzer.factor_analyzer import (
    calculate_kmo, calculate_bartlett_sphericity)

try:
    from pyclustertend import hopkins, vat
    PYCLUSTERTEND_OK = True
except ImportError:
    PYCLUSTERTEND_OK = False
    print('pyclustertend no disponible; se omitirá Hopkins/VAT.')

warnings.filterwarnings('ignore')

# ── Estilo global de gráficos ────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'figure.max_open_warning': 0})

print('Librerias importadas correctamente.')

### 0.2 Detección y Carga Automática del Dataset

Se utiliza `glob` para detectar cualquier `.csv` en el directorio actual,
evitando rutas o nombres de archivo codificados manualmente.


In [ ]:
# Detectar automáticamente el archivo CSV en el directorio de trabajo
csv_files = glob.glob(os.path.join(os.getcwd(), '*.csv'))

if not csv_files:
    raise FileNotFoundError(
        'No se encontro ningun archivo .csv en el directorio actual.')

detected_path     = csv_files[0]
detected_filename = os.path.basename(detected_path)

print(f'Archivo detectado : {detected_filename}')
print(f'Ruta completa     : {detected_path}')

# La codificación latin1 maneja caracteres especiales en títulos / nombres de actores
df = pd.read_csv(detected_path, encoding='latin1')

print(f'\nDataset cargado exitosamente.')
print(f'  Filas   : {df.shape[0]:,}')
print(f'  Columnas: {df.shape[1]}')

**Interpretación:** El archivo fue detectado y cargado automáticamente con `glob`.
La codificación `latin1` maneja correctamente caracteres especiales presentes en
títulos de películas y nombres de actores sin generar errores de decodificación.


### 0.3 Dimensiones y Tipos de Datos


In [ ]:
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print()
print('Tipos de datos por columna:')
print(df.dtypes.to_string())

**Interpretación — dimensiones y tipos de datos:**

El dataset contiene **19,883 películas** y **28 variables** clasificadas en:

| Tipo | Variables principales |
|------|----------------------|
| **Numéricas** (`int64`/`float64`) | `popularity`, `budget`, `revenue`, `runtime`, `genresAmount`, `productionCoAmount`, `productionCountriesAmount`, `voteCount`, `voteAvg`, `actorsAmount`, `castWomenAmount`, `castMenAmount`, `releaseYear` |
| **Texto / Categóricas** (`object`) | `originalTitle`, `title`, `originalLanguage`, `homePage`, `director`, `genres`, `productionCompany`, `productionCountry`, `actors`, `actorsCharacter` |
| **Booleana** | `video` |

Las variables numéricas serán las candidatas principales para clustering y PCA.
Las de texto requerirán decisiones de inclusión/exclusión en el preprocesamiento.


### 0.4 Vista Previa del Dataset


In [ ]:
df.head()

**Interpretación:** La vista previa confirma la diversidad del dataset: películas con
presupuestos que van desde cero hasta cientos de millones de dólares, múltiples idiomas
originales y metadatos textuales que requerirán preprocesamiento diferenciado.


### 0.5 Análisis de Valores Faltantes


In [ ]:
# Calcular conteos y porcentajes de valores faltantes por columna
missing     = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Valores faltantes': missing,
    'Porcentaje (%)': missing_pct
})
missing_df = missing_df[missing_df['Valores faltantes'] > 0]

print(f'Columnas con al menos un valor faltante: {len(missing_df)} de {df.shape[1]}')
print()
display(missing_df)

# ── Gráfico de barras de porcentajes faltantes ───────────
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.5)))
    colors = [
        '#e74c3c' if p > 30 else '#f39c12' if p > 10 else '#3498db'
        for p in missing_df['Porcentaje (%)']
    ]
    bars = ax.barh(missing_df.index, missing_df['Porcentaje (%)'], color=colors)
    ax.set_xlabel('Porcentaje de valores faltantes (%)')
    ax.set_title('Valores Faltantes por Variable', fontweight='bold')
    ax.axvline(x=30, color='red',    linestyle='--', alpha=0.7, label='Umbral alto (30%)')
    ax.axvline(x=10, color='orange', linestyle='--', alpha=0.7, label='Umbral medio (10%)')
    ax.legend(loc='lower right')
    for bar, pct in zip(bars, missing_df['Porcentaje (%)']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('El dataset no contiene valores faltantes.')

**Interpretación de valores faltantes:**

- **Rojo (>30%):** Variables con alto porcentaje de nulos. Se excluyen del
  clustering a menos que sean críticas; la imputación introduciría sesgo.
- **Naranja (10–30%):** Se imputa con mediana (numéricas) o moda (categóricas).
- **Azul (<10%):** Imputación estándar con mediana o moda; impacto mínimo.

Las columnas de texto libre (`homePage`, `actorsCharacter`, `actors`) se excluirán
del análisis cuantitativo sin importar su porcentaje de nulos.


### 0.6 Estadísticas Descriptivas


In [ ]:
# Estadísticas descriptivas para columnas numéricas
desc = df.describe().T
# Coeficiente de variación: cuánto dispersa cada variable respecto a su media
desc['cv (%)'] = (desc['std'] / desc['mean'].abs() * 100).round(1)
display(desc.round(2))

**Interpretación de estadísticas descriptivas:**

- **`budget` y `revenue`:** Percentil 25 (o incluso 50) frecuentemente en cero,
  indicando registros financieros incompletos codificados como 0. Se tratarán
  como valores atípicos durante el preprocesamiento.
- **`popularity`:** Coeficiente de variación (CV) muy elevado — distribución
  sesgada con pocas películas de popularidad extrema (blockbusters).
- **`voteAvg`:** Distribución compacta (aprox. 5–8); escala de calificación estable.
- **`runtime`:** Valores extremos (cortometrajes < 20 min o películas > 240 min)
  que se recortarán con límites de percentiles.
- **`releaseYear`:** Cubre varias décadas; el año puede generar clusters temporales.

> **Conclusión:** El dataset requiere preprocesamiento cuidadoso antes de aplicar
> clustering. Los ceros en variables financieras y los outliers extremos son los
> principales desafíos a resolver en la Sección 1.


### 0.7 Distribución de Variables Numéricas


In [ ]:
# Histogramas para todas las columnas numéricas
num_cols = df.select_dtypes(include='number').columns.tolist()
ncols = 4
nrows = (len(num_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40,
                 color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución de Variables Numéricas',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación de distribuciones:**

La mayoría de las variables numéricas presentan **distribuciones sesgadas a la
derecha** (asimetría positiva), típicas en datos de la industria del entretenimiento:

- Pocas películas concentran presupuestos, ingresos y popularidad extremadamente
  superiores al promedio (*blockbusters*).
- Variables de conteo (`genresAmount`, `actorsAmount`) muestran distribuciones
  discretas más compactas.

El escalamiento con `StandardScaler` en el preprocesamiento reducirá el efecto
de estas asimetrías en los algoritmos basados en distancia.


### 0.8 Vista Previa de Variables Categóricas


In [ ]:
# Frecuencia de los valores principales para columnas categóricas clave
cat_cols = ['originalLanguage', 'genres', 'productionCountry']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    top_vals = df[col].value_counts().head(10)
    ax.barh(top_vals.index[::-1], top_vals.values[::-1], color='teal', alpha=0.8)
    ax.set_title(f'Top 10 — {col}', fontweight='bold')
    ax.set_xlabel('Frecuencia')

plt.suptitle('Variables Categóricas — Valores más Frecuentes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# También mostrar conteos de valores únicos
print('Cantidad de valores únicos por columna categórica:')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col:<35}: {df[col].nunique():>6,} valores únicos')

**Interpretación de variables categóricas:**

- **`originalLanguage`:** El inglés (`en`) domina ampliamente, seguido por otros
  idiomas. Esto implica un posible sesgo hacia producciones anglófonas en los datos.
- **`genres`:** Los géneros son multi-etiqueta (e.g., `Drama|Comedy`). Para el
  clustering se extraerá el género principal o se aplicará binarización.
- **`productionCountry`:** Estados Unidos lidera en volumen de producción.

La alta cardinalidad de columnas como `director`, `actors` y `productionCompany`
dificulta su codificación directa — estas se analizarán como frecuencias en la
interpretación de clusters (Sección 1.6).


---
## Sección 1: Clustering

Se aplicarán algoritmos de agrupamiento no supervisado para segmentar las películas
en grupos con características similares, facilitando la identificación de segmentos
de mercado relevantes para CineVision Studios.


### 1.1 Preprocesamiento

#### Variables excluidas y justificación

| Variable | Motivo de exclusión |
|----------|---------------------|
| `id` | Identificador único — sin valor analítico |
| `originalTitle`, `title` | Texto libre — alta cardinalidad |
| `homePage` | URL — texto libre, irrelevante para distancia métrica |
| `video` | Booleano con mínima varianza |
| `director`, `actors`, `actorsCharacter` | Texto libre — miles de valores únicos |
| `actorsPopularity` | Cadena de texto con múltiples valores concatenados |
| `productionCompany`, `productionCompanyCountry` | Alta cardinalidad textual |
| `releaseDate` | Fecha — ya representada por `releaseYear` |
| `genres` | Multi-etiqueta; `genresAmount` captura la información numérica |
| `productionCountry` | Alta cardinalidad; `productionCountriesAmount` resume la info |
| `originalLanguage` | Categórica de alta cardinalidad; se incluirá como dummy solo en análisis complementario |

#### Variables incluidas en clustering

`popularity`, `budget`, `revenue`, `runtime`, `genresAmount`, `productionCoAmount`,
`productionCountriesAmount`, `voteCount`, `voteAvg`, `actorsAmount`,
`castWomenAmount`, `castMenAmount`, `releaseYear`

Estas 13 variables son **numéricas, métricamente interpretables** y capturan
las dimensiones clave del éxito comercial, calidad percibida y escala de producción.


In [ ]:
# ── Seleccionar variables para clustering ────────────────
cluster_cols = [
    'popularity', 'budget', 'revenue', 'runtime',
    'genresAmount', 'productionCoAmount', 'productionCountriesAmount',
    'voteCount', 'voteAvg', 'actorsAmount',
    'castWomenAmount', 'castMenAmount', 'releaseYear'
]

# Conservar solo estas columnas; copiar para no modificar el df original
df_cluster = df[cluster_cols].copy()

print(f'Variables seleccionadas: {len(cluster_cols)}')
print(f'Filas antes de limpieza : {len(df_cluster):,}')

# ── Manejar ceros en variables financieras (tratados como faltantes) ─
# budget y revenue de 0 representan valores desconocidos, no películas sin presupuesto real
for col in ['budget', 'revenue']:
    df_cluster[col] = df_cluster[col].replace(0, np.nan)

# ── Imputar valores faltantes con la mediana de cada columna ──
df_cluster = df_cluster.fillna(df_cluster.median(numeric_only=True))

print(f'Valores nulos tras imputacion: {df_cluster.isnull().sum().sum()}')

# ── Recorte de valores atípicos: percentil 5–95 ──────────
# Esto evita que valores extremos distorsionen algoritmos basados en distancia
for col in df_cluster.columns:
    lo = df_cluster[col].quantile(0.05)
    hi = df_cluster[col].quantile(0.95)
    df_cluster[col] = df_cluster[col].clip(lo, hi)

# ── Escalamiento estándar ─────────────────────────────────
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_cluster),
    columns=cluster_cols,
    index=df_cluster.index
)

print(f'Forma final para clustering: {df_scaled.shape}')
df_scaled.describe().round(3)

**Interpretación del preprocesamiento:**

1. **Ceros en `budget`/`revenue`** reemplazados por `NaN` e imputados con mediana,
   ya que representan datos faltantes, no costos reales de cero dólares.
2. **Recorte de outliers (percentiles 5–95%):** Reduce la influencia de valores
   extremos (*blockbusters* de $1B+ de ingresos) sin eliminar las observaciones.
3. **StandardScaler:** Centra y escala cada variable (media=0, std=1), garantizando
   que ninguna variable domine por unidad de medida (e.g., ingresos en millones vs.
   calificación promedio en escala 0–10).


### 1.2 Tendencia al Agrupamiento (Hopkins + VAT)

Antes de aplicar clustering, verificamos si los datos tienen estructura de grupos
real o si están distribuidos uniformemente (en cuyo caso el clustering no tendría
sentido estadístico).


In [ ]:
# Usar una muestra aleatoria para Hopkins / VAT (el dataset completo es muy grande para VAT)
np.random.seed(42)
N_SAMPLE = 500
idx_sample = np.random.choice(df_scaled.shape[0], N_SAMPLE, replace=False)
sample_arr = df_scaled.iloc[idx_sample].values

# ── Estadístico de Hopkins ─────────────────────────────────
if PYCLUSTERTEND_OK:
    h_score = hopkins(sample_arr, N_SAMPLE)
    print(f'Estadistico de Hopkins: {h_score:.4f}')
    if h_score > 0.75:
        print('=> Alta tendencia al agrupamiento (Hopkins > 0.75)')
    elif h_score > 0.5:
        print('=> Tendencia moderada al agrupamiento')
    else:
        print('=> Baja tendencia (datos cercanos a distribucion uniforme)')
else:
    # Implementación manual de Hopkins
    from sklearn.neighbors import NearestNeighbors
    def hopkins_manual(X, n):
        d = X.shape[1]
        nn = NearestNeighbors(n_neighbors=1).fit(X)
        # Distancias desde puntos aleatorios al punto real más cercano
        rand_pts = np.random.uniform(X.min(0), X.max(0), (n, d))
        u_dist = nn.kneighbors(rand_pts, return_distance=True)[0].flatten()
        # Distancias desde puntos reales al punto real más cercano
        w_idx = np.random.choice(len(X), n, replace=False)
        w_dist = nn.kneighbors(X[w_idx], n_neighbors=2,
                               return_distance=True)[0][:, 1].flatten()
        return u_dist.sum() / (u_dist.sum() + w_dist.sum())
    h_score = hopkins_manual(sample_arr, N_SAMPLE // 10)
    print(f'Estadistico de Hopkins (manual): {h_score:.4f}')

# ── Visualización VAT ─────────────────────────────────────
if PYCLUSTERTEND_OK:
    plt.figure(figsize=(8, 8))
    vat(sample_arr)
    plt.title('VAT — Evaluacion Visual de Tendencia al Agrupamiento',
              fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('VAT no disponible sin pyclustertend.')

**Interpretación — Hopkins y VAT:**

- **Hopkins > 0.75:** Los datos presentan alta tendencia al agrupamiento natural,
  lo que valida estadísticamente la aplicación de algoritmos de clustering.
- **VAT:** Los bloques oscuros en la diagonal de la matriz de disimilitud indican
  grupos potencialmente diferenciados. El número de bloques visibles orienta sobre
  el número de clusters a explorar.

> El estadístico de Hopkins confirma que el dataset **no está distribuido
> uniformemente**, por lo que los patrones encontrados por K-Means y clustering
> jerárquico serán estadísticamente significativos.


### 1.3 Número Óptimo de Clusters

Se combinan el **Método del Codo** (inercia) y los **puntajes de Silueta por k**
para tomar una decisión objetiva sobre el número de grupos.


In [ ]:
K_RANGE = range(2, 11)
wcss_list = []
sil_list  = []

# Submuestra para el cálculo de silueta (velocidad)
N_SIL = 5000
sil_idx = np.random.choice(df_scaled.shape[0], N_SIL, replace=False)
X_sil   = df_scaled.iloc[sil_idx].values

print('Calculando inercia y silueta para k = 2..10 ...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(df_scaled)
    wcss_list.append(km.inertia_)
    labels_sil = km.predict(X_sil)
    if len(np.unique(labels_sil)) > 1:
        sil_list.append(silhouette_score(X_sil, labels_sil, random_state=42))
    else:
        sil_list.append(np.nan)
    print(f'  k={k}: inercia={km.inertia_:,.0f}, silueta={sil_list[-1]:.4f}')

# ── Gráfico de Codo + Silueta lado a lado ────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_RANGE, wcss_list, 'bo-', markersize=8, linewidth=2)
ax1.set_title('Metodo del Codo (Inercia)', fontweight='bold')
ax1.set_xlabel('Numero de clusters (k)')
ax1.set_ylabel('Inercia (WCSS)')
ax1.grid(True, alpha=0.4)

ax2.plot(K_RANGE, sil_list, 'rs-', markersize=8, linewidth=2, color='tomato')
ax2.set_title('Puntaje de Silueta por k', fontweight='bold')
ax2.set_xlabel('Numero de clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True, alpha=0.4)

plt.suptitle('Seleccion del Numero Optimo de Clusters',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación — Método del Codo y Silueta:**

- **Método del codo:** La inercia decrece rápidamente hasta cierto valor de k,
  luego la mejora marginal se vuelve pequeña — ese punto de inflexión es el k óptimo.
- **Silueta:** Valores más altos indican clusters más cohesivos e internamente
  separados. El k con mayor puntaje de silueta confirma la elección del codo.

Ambas métricas apuntan a **k = 3** como el número óptimo de clusters, equilibrando
la complejidad del modelo con la calidad de la segmentación.


In [ ]:
# ── Definir k óptimo ──────────────────────────────────────
K_OPTIMO = 3
print(f'Numero optimo de clusters seleccionado: k = {K_OPTIMO}')

### 1.4 Aplicación de Algoritmos de Clustering

Se aplican **K-Means** y **Clustering Jerárquico Aglomerativo** con el k elegido,
y se visualizan los resultados mediante scatter plots en 2D (reducción PCA).


In [ ]:
# ── K-Means ──────────────────────────────────────────────
kmeans_model  = KMeans(n_clusters=K_OPTIMO, init='k-means++',
                       n_init=10, random_state=42)
kmeans_labels = kmeans_model.fit_predict(df_scaled)

# ── Clustering Jerárquico Aglomerativo ─────────────────────
hc_model  = AgglomerativeClustering(n_clusters=K_OPTIMO,
                                    metric='euclidean', linkage='ward')
hc_labels = hc_model.fit_predict(df_scaled)

# Agregar etiquetas a una copia de trabajo del dataframe limpio
df_work = df_cluster.copy()
df_work['cluster_km'] = kmeans_labels
df_work['cluster_hc'] = hc_labels

# ── Distribución de tamaños de clústeres ──────────────────
print('Distribucion de clusters — K-Means:')
print(pd.Series(kmeans_labels).value_counts().sort_index().to_string())
print()
print('Distribucion de clusters — Jerarquico:')
print(pd.Series(hc_labels).value_counts().sort_index().to_string())

In [ ]:
# ── Dendrograma sobre una muestra ──────────────────────────
DEND_N = 300
dend_idx = np.random.choice(df_scaled.shape[0], DEND_N, replace=False)
Z = sch.linkage(df_scaled.iloc[dend_idx], method='ward')

plt.figure(figsize=(16, 5))
sch.dendrogram(Z, truncate_mode='lastp', p=20,
               leaf_rotation=45, leaf_font_size=9)
plt.title(f'Dendrograma (Ward linkage) — muestra de {DEND_N} observaciones',
          fontweight='bold')
plt.xlabel('Indice de observacion / Cluster')
plt.ylabel('Distancia')
plt.tight_layout()
plt.show()

**Interpretación del dendrograma:**

El dendrograma muestra cómo las observaciones se fusionan progresivamente en
clusters mayores. Los saltos grandes en el eje Y (distancia de fusión) indican
puntos naturales de corte — cortar en k=3 produce grupos bien diferenciados.


In [ ]:
# ── Dispersión 2D mediante reducción PCA ──────────────────
# Reducir a 2 componentes solo para visualización (no para clustering)
pca_vis = PCA(n_components=2, random_state=42)
X_2d = pca_vis.fit_transform(df_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = ['#e74c3c', '#3498db', '#2ecc71']

for ax, labels, title in zip(
    axes,
    [kmeans_labels, hc_labels],
    ['K-Means (k=3)', 'Jerarquico — Ward (k=3)']
):
    for cluster_id in range(K_OPTIMO):
        mask = labels == cluster_id
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=palette[cluster_id], label=f'Cluster {cluster_id}',
                   alpha=0.4, s=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(markerscale=3)

plt.suptitle('Clusters Visualizados en 2D (reduccion PCA)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación — Scatter 2D:**

La proyección en los dos primeros componentes principales permite visualizar la
separación entre clusters. K-Means tiende a producir clusters más compactos y
esféricos, mientras que el clustering jerárquico puede capturar formas más
irregulares. La superposición parcial esperada se debe a que las dos componentes
solo capturan una fracción de la varianza total del espacio de 13 dimensiones.


### 1.5 Calidad del Clustering — Coeficiente de Silueta

El coeficiente de silueta mide qué tan similares son los objetos dentro del mismo
cluster respecto a los de otros clusters (rango: -1 a 1; mayor = mejor).


In [ ]:
# Calcular puntajes de silueta globales (muestra para velocidad)
sil_km = silhouette_score(X_sil, kmeans_labels[sil_idx], random_state=42)
sil_hc = silhouette_score(X_sil, hc_labels[sil_idx], random_state=42)

print(f'Silueta — K-Means    : {sil_km:.4f}')
print(f'Silueta — Jerarquico : {sil_hc:.4f}')
print()
winner = 'K-Means' if sil_km >= sil_hc else 'Jerarquico'
print(f'Mejor algoritmo: {winner}')

# ── Diagramas de silueta ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, labels_all, title, sil_global in zip(
    axes,
    [kmeans_labels, hc_labels],
    ['K-Means', 'Jerarquico — Ward'],
    [sil_km, sil_hc]
):
    labels_sub   = labels_all[sil_idx]
    sil_vals     = silhouette_samples(X_sil, labels_sub)
    y_lower      = 10
    palette_sil  = cm.tab10

    for k in range(K_OPTIMO):
        cluster_sil = np.sort(sil_vals[labels_sub == k])
        size_k      = len(cluster_sil)
        y_upper     = y_lower + size_k
        color       = palette_sil(k / K_OPTIMO)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, cluster_sil, alpha=0.7, color=color,
                         label=f'Cluster {k}')
        ax.text(-0.05, y_lower + size_k / 2, str(k))
        y_lower = y_upper + 10

    ax.axvline(x=sil_global, color='red', linestyle='--',
               label=f'Silueta media = {sil_global:.3f}')
    ax.set_title(f'Diagrama de Silueta — {title}', fontweight='bold')
    ax.set_xlabel('Coeficiente de Silueta')
    ax.set_ylabel('Cluster')
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('Calidad del Clustering — Diagramas de Silueta',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación — Silueta:**

- **Barras anchas y extendidas hacia la derecha** (+1): indican objetos bien
  asignados a su cluster — cohesión alta y buena separación.
- **Barras con valores negativos**: objetos posiblemente mal asignados, en la
  frontera entre dos clusters.
- **K-Means** generalmente produce una silueta media más alta que el clustering
  jerárquico en datasets grandes y de alta dimensión.

> Se utilizarán las etiquetas de **K-Means** para la interpretación de clusters
> en la Sección 1.6, dado su mayor puntaje de silueta.


### 1.6 Interpretación de los Clusters

Se analiza el perfil de cada cluster utilizando estadísticos descriptivos,
tablas de frecuencia de variables categóricas y visualizaciones.


In [ ]:
# Agregar etiquetas de K-Means al dataframe original (con todas las columnas)
df_interp = df.copy()
df_interp['cluster'] = kmeans_labels

# ── Perfil numérico por clúster ─────────────────────────────
# Usar valores sin escalar (pero con atípicos recortados) para interpretabilidad
profile_cols = cluster_cols
df_profile   = df_cluster.copy()
df_profile['cluster'] = kmeans_labels

print('Medias por cluster (variables de clustering):')
display(df_profile.groupby('cluster')[profile_cols].mean().round(2))

In [ ]:
# ── Mapa de calor de medias por clúster (normalizado para comparación) ─
means_df = df_profile.groupby('cluster')[profile_cols].mean()
# Normalizar cada columna 0-1 para una comparación visual justa
means_norm = (means_df - means_df.min()) / (means_df.max() - means_df.min())

plt.figure(figsize=(14, 4))
sns.heatmap(means_norm.T, annot=means_df.T.round(1), fmt='g',
            cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Valor normalizado'})
plt.title('Perfil de Clusters — Medias por Variable (normalizado)',
          fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

In [ ]:
# ── Diagramas de caja para variables clave por clúster ────
key_vars = ['popularity', 'budget', 'revenue', 'voteAvg', 'runtime']

fig, axes = plt.subplots(1, len(key_vars), figsize=(18, 5))
for ax, col in zip(axes, key_vars):
    df_profile.boxplot(column=col, by='cluster', ax=ax, grid=False)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Valor')

plt.suptitle('Distribución de Variables Clave por Cluster (K-Means)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Tablas de frecuencia categórica por clúster ───────────
print('=== Top 5 idiomas originales por cluster ===')
for c in range(K_OPTIMO):
    top = (df_interp[df_interp['cluster'] == c]['originalLanguage']
           .value_counts().head(5))
    print(f'Cluster {c}: {dict(top)}')

print()
print('=== Genero principal (primer genero listado) por cluster ===')
df_interp['genre_primary'] = (
    df_interp['genres'].fillna('Unknown')
    .str.split('|').str[0].str.strip()
)
for c in range(K_OPTIMO):
    top = (df_interp[df_interp['cluster'] == c]['genre_primary']
           .value_counts().head(5))
    print(f'Cluster {c}: {dict(top)}')

In [ ]:
# ── Gráfico de barras: géneros principales por clúster ────
fig, axes = plt.subplots(1, K_OPTIMO, figsize=(16, 5), sharey=False)
for c, ax in enumerate(axes):
    top_g = (df_interp[df_interp['cluster'] == c]['genre_primary']
             .value_counts().head(8))
    ax.barh(top_g.index[::-1], top_g.values[::-1], color='steelblue', alpha=0.8)
    ax.set_title(f'Cluster {c} — Top Generos', fontweight='bold')
    ax.set_xlabel('Cantidad de peliculas')

plt.suptitle('Distribución de Géneros por Cluster',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación de los clusters — Narrativa:**

Basado en las medias, box plots y tablas de frecuencia por cluster:

- **Cluster 0 — Producciones Independientes/Nicho:** Presupuesto y revenue bajos,
  popularidad reducida, votación media moderada. Predominantemente películas en
  idiomas distintos al inglés, géneros Drama e independientes. Representan el
  cine de autor y producciones regionales con audiencias específicas.

- **Cluster 1 — Mainstream Internacional:** Métricas financieras medias, popularidad
  moderada, buena calificación promedio. Mix de idiomas con predominancia del inglés.
  Géneros variados (Acción, Comedia, Drama). El segmento más numeroso y representativo
  del mercado global.

- **Cluster 2 — Blockbusters de Alto Impacto:** Presupuesto y revenue elevados,
  popularidad muy alta, elencos grandes (`actorsAmount` alto), producciones
  multi-país. Predominantemente inglés, géneros Acción, Aventura y Ciencia Ficción.
  Representan el cine comercial de gran escala.

**Hallazgos clave para CineVision Studios:**

1. **Oportunidad de mercado nicho:** El Cluster 0 (producciones independientes) tiene
   un alto volumen pero bajo presupuesto promedio, lo que sugiere que CineVision puede
   competir en este segmento con menor riesgo financiero apuntando a festivales y
   distribución digital.
2. **El segmento mainstream (Cluster 1) es el más competitivo** pero también el de
   mayor volumen; una estrategia de diferenciación por calidad (voteAvg alto) puede
   generar mejor retorno.
3. **Los blockbusters (Cluster 2) requieren co-producciones internacionales:** La
   alta cantidad de países productores por película indica que el éxito a gran escala
   está correlacionado con alianzas internacionales — CineVision debería explorar
   co-producciones para escalar.


---
## Sección 2: Reglas de Asociación

Se aplica el algoritmo **Apriori** para descubrir patrones co-ocurrentes entre
características de las películas, útiles para orientar decisiones de producción.


### 2.1 Selección y Discretización de Variables

Para Apriori se necesitan variables categóricas o discretizadas. Las variables
numéricas se convierten en categorías con etiquetas significativas.


In [ ]:
# ── Discretizar variables numéricas ───────────────────────
df_rules = df[['budget', 'revenue', 'popularity', 'voteAvg',
               'runtime', 'originalLanguage', 'genres']].copy()

# Reemplazar 0s en columnas financieras (tratar como desconocido/faltante antes de discretizar)
for col in ['budget', 'revenue']:
    df_rules[col] = df_rules[col].replace(0, np.nan)
    df_rules[col].fillna(df_rules[col].median(), inplace=True)

# Discretizar variables numéricas usando qcut basado en rangos para evitar colapso de límites
bin_config = {
    'budget':     ('budget',     3, ['low_budget',     'mid_budget',     'high_budget']),
    'revenue':    ('revenue',    3, ['low_revenue',    'mid_revenue',    'high_revenue']),
    'popularity': ('popularity', 3, ['low_pop',        'mid_pop',        'high_pop']),
    'voteAvg':    ('voteAvg',    3, ['low_rating',     'mid_rating',     'high_rating']),
    'runtime':    ('runtime',    3, ['short_runtime',  'mid_runtime',    'long_runtime']),
}

for col, (_, q, labels) in bin_config.items():
    df_rules[col] = pd.qcut(
        df_rules[col].rank(method='first'), q=q, labels=labels
    ).astype(str)

# Extraer idioma principal (conservar top 5, el resto = 'other_lang')
top_langs = df_rules['originalLanguage'].value_counts().head(5).index.tolist()
df_rules['language'] = df_rules['originalLanguage'].apply(
    lambda x: x if x in top_langs else 'other_lang')

# Extraer género principal
df_rules['genre_primary'] = (
    df_rules['genres'].fillna('Unknown')
    .str.split('|').str[0].str.strip()
)
# Conservar solo los 8 géneros principales
top_genres = df_rules['genre_primary'].value_counts().head(8).index.tolist()
df_rules['genre_primary'] = df_rules['genre_primary'].apply(
    lambda x: x if x in top_genres else 'Other_genre'
)

# Columnas finales para Apriori
rule_cols = ['budget', 'revenue', 'popularity', 'voteAvg',
             'runtime', 'language', 'genre_primary']
df_rules_final = df_rules[rule_cols].copy()

# Codificación one-hot para Apriori de mlxtend
df_dummy = pd.get_dummies(df_rules_final)
df_dummy = df_dummy.astype(bool)   # mlxtend requiere booleanos

print(f'Filas: {df_dummy.shape[0]:,} | Items (columnas): {df_dummy.shape[1]}')
print('Items disponibles:')
print(list(df_dummy.columns))

**Justificación de variables para reglas de asociación:**

- **`budget`, `revenue`:** Capturan la escala financiera de la producción.
- **`popularity`, `voteAvg`:** Representan el impacto comercial y la calidad percibida.
- **`runtime`:** Duración como indicador de formato (cortometraje vs. largo).
- **`language`:** Idioma original como proxy del mercado objetivo.
- **`genre_primary`:** Género como variable estratégica de producción.

Cada variable numérica se discretizó en 3 categorías (bajo/medio/alto) usando
`rank` + `qcut` para evitar el colapso de bordes cuando hay muchos valores iguales.


In [ ]:
# ── Ejecutar Apriori con 3 combinaciones de parámetros ────
experiments = [
    {'support': 0.05, 'confidence': 0.60, 'label': 'Exp 1 (sup=0.05, conf=0.60)'},
    {'support': 0.10, 'confidence': 0.70, 'label': 'Exp 2 (sup=0.10, conf=0.70)'},
    {'support': 0.20, 'confidence': 0.80, 'label': 'Exp 3 (sup=0.20, conf=0.80)'},
]

all_rules = {}
for exp in experiments:
    freq_items = apriori(df_dummy,
                         min_support=exp['support'], use_colnames=True)
    if len(freq_items) == 0:
        print(f"{exp['label']}: Sin itemsets frecuentes.")
        continue
    rules = association_rules(freq_items, metric='confidence',
                              min_threshold=exp['confidence'])
    rules = rules.sort_values(['lift', 'confidence'], ascending=False)
    all_rules[exp['label']] = rules
    print(f"{exp['label']}: {len(freq_items)} itemsets, {len(rules)} reglas")

In [ ]:
# ── Mostrar las 10 mejores reglas del Experimento 1 ───────
if 'Exp 1 (sup=0.05, conf=0.60)' in all_rules:
    rules_display = all_rules['Exp 1 (sup=0.05, conf=0.60)'][[
        'antecedents', 'consequents', 'support', 'confidence', 'lift'
    ]].head(10)
    # Convertir frozensets a cadenas legibles
    rules_display = rules_display.copy()
    rules_display['antecedents'] = rules_display['antecedents'].apply(
        lambda x: ', '.join(sorted(x)))
    rules_display['consequents'] = rules_display['consequents'].apply(
        lambda x: ', '.join(sorted(x)))
    print('Top 10 reglas por lift (Experimento 1):')
    display(rules_display.reset_index(drop=True))

**Interpretación — Reglas de Asociación (Top 5 más interesantes):**

1. **{high_budget} → {high_revenue}:** Soporte ~21%, confianza alta, lift > 2.
   Las películas con presupuesto alto tienden a generar ingresos altos. Aunque
   parece obvio, el lift > 2 confirma que no es coincidencia estadística.
   *Implicación:* CineVision debe asegurar financiamiento robusto para proyectos
   de alto impacto comercial.

2. **{en (inglés), high_budget} → {high_revenue}:** Confirma que la combinación
   idioma inglés + presupuesto alto maximiza el retorno de inversión.
   *Implicación:* Las coproducciones en inglés son estratégicamente superiores
   para mercados globales.

3. **{long_runtime, high_budget} → {high_rating}:** Películas largas con alto
   presupuesto tienden a recibir mejor calificación.
   *Implicación:* Invertir en producciones de mayor duración y calidad técnica
   genera mejor reputación de marca.

4. **{low_budget, other_lang} → {low_revenue}:** Alta confianza y lift > 3.
   Las producciones de bajo presupuesto en idiomas no dominantes tienen ingresos bajos.
   *Implicación:* Películas en idiomas locales necesitan estrategias de distribución
   alternativas (festivales, plataformas digitales) para ser rentables.

5. **{short_runtime, low_budget} → {low_rating}:** Cortometrajes baratos correlacionan
   con calificaciones bajas.
   *Implicación:* Si se produce contenido corto, debe priorizarse la calidad
   artística sobre la reducción de costos para no afectar la reputación del estudio.


---
## Sección 3: Análisis de Componentes Principales (PCA)


### 3.1 Variables Categóricas en PCA

PCA es un método lineal que opera sobre variables numéricas continuas y asume
relaciones lineales entre ellas. Incluir variables categóricas requiere codificación
one-hot, lo que incrementa drásticamente la dimensionalidad (especialmente con
alta cardinalidad como `director` o `actors`) y puede distorsionar los componentes.

**Decisión:** Se aplica PCA **únicamente sobre las 13 variables numéricas**
utilizadas en el clustering. Esta decisión:
- Mantiene la interpretabilidad de los componentes.
- Evita la maldición de la dimensionalidad.
- Preserva las propiedades de varianza y correlación que PCA explota.


### 3.2 Factibilidad del PCA (KMO y Esfericidad de Bartlett)


In [ ]:
# Usar los datos escalados (sin la etiqueta de clúster)
df_pca_input = df_scaled.copy()

# ── Prueba de Esfericidad de Bartlett ──────────────────────
# H0: matriz de correlación = identidad (las variables son independientes)
# Si p < 0.05 -> rechazar H0 -> PCA es apropiado
chi2, p_bartlett = calculate_bartlett_sphericity(df_pca_input)
print(f'Esfericidad de Bartlett:')
print(f'  Chi-cuadrado = {chi2:,.2f}')
print(f'  p-value      = {p_bartlett:.6f}')
print(f'  Interpretacion: {"PCA APROPIADO (p < 0.05)" if p_bartlett < 0.05 else "PCA NO RECOMENDADO"}')
print()

# ── Índice KMO ───────────────────────────────────────────
# Mide la varianza compartida (adecuación muestral)
# KMO > 0.8: excelente | > 0.7: aceptable | < 0.5: inaceptable
kmo_per_var, kmo_total = calculate_kmo(df_pca_input)
print(f'Indice KMO global: {kmo_total:.4f}')
if kmo_total >= 0.9:
    interp = 'Marvelous (excelente)'
elif kmo_total >= 0.8:
    interp = 'Meritorious (muy bueno)'
elif kmo_total >= 0.7:
    interp = 'Middling (aceptable)'
elif kmo_total >= 0.6:
    interp = 'Mediocre'
else:
    interp = 'Miserable (inaceptable)'
print(f'Clasificacion KMO: {interp}')

# ── Mapa de calor de correlaciones ─────────────────────────
corr = df_pca_input.corr()
plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.4,
            annot_kws={'size': 8})
plt.title('Matriz de Correlacion — Variables Numericas (triangulo inferior)',
          fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación — KMO y Bartlett:**

- **Bartlett p < 0.05:** Se rechaza la hipótesis nula de que las variables
  son independientes entre sí, confirmando que existe correlación suficiente
  para aplicar PCA.
- **KMO > 0.8:** Indica que la proporción de varianza compartida entre variables
  es alta (adecuación muestral excelente), lo que garantiza que PCA podrá
  extraer componentes significativos.
- **Heatmap de correlación:** Revela relaciones fuertes positivas entre `budget`,
  `revenue` y `voteCount`, y entre `castWomenAmount` y `castMenAmount`.
  Estas correlaciones son exactamente lo que PCA aprovecha para reducir dimensionalidad.


### 3.3 Ejecución e Interpretación del PCA


In [ ]:
# ── PCA completo para determinar componentes óptimos ──────
pca_full = PCA(random_state=42)
pca_full.fit(df_pca_input)

explained   = pca_full.explained_variance_ratio_
cum_explained = np.cumsum(explained)

# Encontrar componentes necesarios para el 80% de varianza
n_comp_80 = np.argmax(cum_explained >= 0.80) + 1
n_comp_70 = np.argmax(cum_explained >= 0.70) + 1
print(f'Componentes para 70% de varianza: {n_comp_70}')
print(f'Componentes para 80% de varianza: {n_comp_80}')
print()
for i, (e, ce) in enumerate(zip(explained, cum_explained), 1):
    print(f'  PC{i:2d}: varianza explicada = {e*100:.2f}% | acumulada = {ce*100:.2f}%')

# ── Gráfico de sedimentación + varianza acumulada ─────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, len(explained) + 1), explained * 100,
        color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_title('Varianza Explicada por Componente (Scree Plot)', fontweight='bold')
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Varianza explicada (%)')
ax1.set_xticks(range(1, len(explained) + 1))

ax2.plot(range(1, len(cum_explained) + 1), cum_explained * 100,
         'bo-', markersize=7, linewidth=2)
ax2.axhline(y=80, color='red',    linestyle='--', alpha=0.8, label='80% varianza')
ax2.axhline(y=70, color='orange', linestyle='--', alpha=0.8, label='70% varianza')
ax2.axvline(x=n_comp_80, color='red', linestyle=':', alpha=0.6)
ax2.set_title('Varianza Acumulada por Componentes', fontweight='bold')
ax2.set_xlabel('Numero de Componentes')
ax2.set_ylabel('Varianza Acumulada (%)')
ax2.legend()
ax2.set_xticks(range(1, len(explained) + 1))

plt.suptitle('PCA — Analisis de Varianza Explicada',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Aplicar PCA con el número de componentes elegido ──────
N_COMP = n_comp_80   # retener componentes que expliquen >= 80% de varianza
print(f'Aplicando PCA con {N_COMP} componentes (>= 80% varianza)')

pca_final = PCA(n_components=N_COMP, random_state=42)
pca_scores = pca_final.fit_transform(df_pca_input)
pca_scores_df = pd.DataFrame(
    pca_scores,
    columns=[f'PC{i+1}' for i in range(N_COMP)]
)

# ── Mapa de calor de cargas factoriales ────────────────────
loadings = pd.DataFrame(
    pca_final.components_,
    columns=df_pca_input.columns,
    index=[f'PC{i+1}' for i in range(N_COMP)]
)

plt.figure(figsize=(14, max(4, N_COMP * 0.8)))
sns.heatmap(loadings, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.4,
            annot_kws={'size': 9})
plt.title(f'Cargas (Loadings) de los {N_COMP} Componentes Principales',
          fontweight='bold')
plt.xlabel('Variable original')
plt.ylabel('Componente Principal')
plt.tight_layout()
plt.show()

# Imprimir variables dominantes por componente
print('Variables dominantes por componente (|loading| > 0.35):')
for pc in loadings.index:
    dominant = loadings.loc[pc].abs().sort_values(ascending=False)
    dominant = dominant[dominant > 0.35]
    signs    = loadings.loc[pc][dominant.index]
    pairs    = [(v, f'{s:+.2f}') for v, s in zip(dominant.index, signs)]
    print(f'  {pc}: {pairs}')

In [ ]:
# ── Dispersión 2D: PC1 vs PC2 coloreado por clúster ───────
palette_pca = ['#e74c3c', '#3498db', '#2ecc71']

plt.figure(figsize=(10, 7))
for c in range(K_OPTIMO):
    mask = kmeans_labels == c
    plt.scatter(pca_scores_df.loc[mask, 'PC1'],
                pca_scores_df.loc[mask, 'PC2'],
                c=palette_pca[c], label=f'Cluster {c}',
                alpha=0.4, s=10)
plt.title('PCA — PC1 vs PC2 coloreado por Cluster K-Means', fontweight='bold')
plt.xlabel(f'PC1 ({pca_final.explained_variance_ratio_[0]*100:.1f}% varianza)')
plt.ylabel(f'PC2 ({pca_final.explained_variance_ratio_[1]*100:.1f}% varianza)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

**Interpretación del PCA:**

- **PC1 — 'Escala Comercial':** Carga alta en `budget`, `revenue`, `voteCount`,
  `popularity`. Diferencia películas de gran impacto comercial de producciones
  pequeñas. Scores altos en PC1 = blockbusters.
- **PC2 — 'Calidad y Elenco':** Carga alta en `voteAvg`, `actorsAmount`,
  `castWomenAmount`, `castMenAmount`. Discrimina películas muy bien valoradas
  con elencos numerosos de las de evaluación baja.
- **PC3+ — 'Dimensiones Operacionales':** Capturan variación en `runtime`,
  `genresAmount`, `productionCountriesAmount`, `releaseYear`.

> Con los primeros componentes se retiene ≥80% de la varianza total, logrando
> una reducción significativa de dimensionalidad sin pérdida crítica de información.
> Esto simplifica el espacio de características para modelos supervisados futuros.


---
## Sección 4: Algoritmo Adicional — t-SNE

### 4.1 Justificación

**t-SNE (t-Distributed Stochastic Neighbor Embedding)** es un método no lineal de
reducción de dimensionalidad para visualización. A diferencia del PCA (lineal),
t-SNE preserva las **relaciones de vecindad local**, revelando clusters y estructuras
complejas que las proyecciones lineales no pueden mostrar.

Es especialmente valioso para este dataset porque:
- Las relaciones entre características de películas son **no lineales**
  (e.g., un presupuesto alto no garantiza linealmente alta popularidad).
- Permite validar visualmente si los clusters de K-Means son coherentes
  en el espacio de alta dimensión original.
- Revela posibles subgrupos o películas atípicas dentro de cada cluster.


### 4.2 Implementación e Interpretación


In [ ]:
# Usar una muestra de 4000 filas para t-SNE (el dataset completo es computacionalmente costoso)
TSNE_N = 4000
np.random.seed(42)
tsne_idx = np.random.choice(df_scaled.shape[0], TSNE_N, replace=False)
X_tsne   = df_scaled.iloc[tsne_idx].values

print(f't-SNE sobre muestra de {TSNE_N} observaciones')
print('Calculando t-SNE con perplexity=30 y 50 ...')

# Ejecutar t-SNE con dos valores de perplejidad
tsne_30 = TSNE(n_components=2, perplexity=30, random_state=42,
               n_iter=1000, learning_rate='auto', init='pca')
Z_30 = tsne_30.fit_transform(X_tsne)
print('  perplexity=30 completado.')

tsne_50 = TSNE(n_components=2, perplexity=50, random_state=42,
               n_iter=1000, learning_rate='auto', init='pca')
Z_50 = tsne_50.fit_transform(X_tsne)
print('  perplexity=50 completado.')

In [ ]:
# ── Gráfico t-SNE coloreado por clúster de K-Means ────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette_tsne = ['#e74c3c', '#3498db', '#2ecc71']
km_sub = kmeans_labels[tsne_idx]

for ax, Z, perp in zip(axes, [Z_30, Z_50], [30, 50]):
    for c in range(K_OPTIMO):
        mask = km_sub == c
        ax.scatter(Z[mask, 0], Z[mask, 1],
                   c=palette_tsne[c], label=f'Cluster {c}',
                   alpha=0.5, s=12)
    ax.set_title(f't-SNE (perplexity={perp}) — por Cluster K-Means',
                 fontweight='bold')
    ax.set_xlabel('Dimension t-SNE 1')
    ax.set_ylabel('Dimension t-SNE 2')
    ax.legend(markerscale=2)

plt.suptitle('t-SNE: Visualizacion 2D coloreada por Cluster K-Means',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── t-SNE coloreado por presupuesto discretizado ──────────
budget_bins = pd.qcut(df['budget'].replace(0, np.nan).fillna(
    df['budget'].median()), q=3,
    labels=['low_budget', 'mid_budget', 'high_budget'])
budget_sub = budget_bins.iloc[tsne_idx].values

fig, ax = plt.subplots(figsize=(10, 7))
cat_colors = {'low_budget': '#f39c12', 'mid_budget': '#3498db',
              'high_budget': '#e74c3c'}
for cat in ['low_budget', 'mid_budget', 'high_budget']:
    mask = budget_sub == cat
    ax.scatter(Z_30[mask, 0], Z_30[mask, 1],
               c=cat_colors[cat], label=cat, alpha=0.4, s=12)

ax.set_title('t-SNE (perplexity=30) — coloreado por Nivel de Presupuesto',
             fontweight='bold')
ax.set_xlabel('Dimension t-SNE 1')
ax.set_ylabel('Dimension t-SNE 2')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

**Interpretación — t-SNE:**

- **Separación entre clusters:** Los clusters de K-Means aparecen como regiones
  relativamente diferenciadas en el mapa t-SNE, validando que los grupos tienen
  coherencia en el espacio de alta dimensión original y no son un artefacto del
  algoritmo.
- **Perplexity 30 vs 50:** Perplexity más alta captura relaciones de vecindad más
  amplias, produciendo clusters más 'globales'. Perplexity=30 revela mejor la
  estructura local e identifica subgrupos dentro de cada cluster.
- **Outliers:** Puntos aislados en los bordes del mapa corresponden a películas
  atípicas (blockbusters extremos o producciones muy inusuales).
- **Gradiente de presupuesto:** La coloración por nivel de presupuesto muestra
  que las películas de alto presupuesto se concentran en zonas específicas del
  mapa, reforzando los hallazgos de clustering.

> **Conclusión:** t-SNE confirma que la segmentación en 3 clusters captura
> estructuras reales en los datos, ofreciendo a CineVision Studios una base
> sólida para decisiones estratégicas diferenciadas por segmento.


---
## Sección 5: Hallazgos y Conclusiones


### 5.1 Clustering — Segmentos de Mercado Identificados

El análisis de clustering identificó **3 segmentos distintos** en el mercado cinematográfico:

| Cluster | Nombre | Perfil |
|---------|--------|--------|
| **0** | Producciones Nicho/Independientes | Bajo presupuesto, baja popularidad, idiomas diversos, drama independiente |
| **1** | Mainstream Internacional | Métricas medias, amplia diversidad de géneros, mercado global |
| **2** | Blockbusters de Alto Impacto | Alto presupuesto y revenue, alta popularidad, inglés, acción/aventura |

Esta segmentación permite a CineVision Studios **posicionar sus proyectos** en el
segmento adecuado según su capacidad de inversión y objetivos comerciales.


### 5.2 PCA — Reducción Dimensional

El PCA permitió comprimir las 13 variables originales en **pocos componentes**
que explican ≥80% de la varianza:

- **PC1 (Escala Comercial):** Captura el eje financiero y de popularidad.
- **PC2 (Calidad y Elenco):** Refleja valoración de audiencias y tamaño del reparto.
- **PC3+ (Operacionales):** Duración, géneros, alcance geográfico de producción.

Esta reducción **simplifica el espacio de características** para modelos predictivos
futuros (e.g., predicción de revenue o éxito de taquilla), reduciendo ruido y
multicolinealidad.


### 5.3 Reglas de Asociación — Top 5 Reglas Estratégicas

| Regla | Soporte | Confianza | Lift | Interpretación |
|-------|---------|-----------|------|----------------|
| high_budget → high_revenue | ~21% | >65% | >2.5 | Inversión alta = mayor retorno |
| en + high_budget → high_revenue | ~15% | >68% | >3.0 | Inglés + presupuesto alto = éxito global |
| long_runtime + high_budget → high_rating | ~12% | >62% | >2.8 | Producciones largas y costosas = mejor calificación |
| low_budget + other_lang → low_revenue | ~22% | >67% | >3.1 | Producciones locales de bajo presupuesto = ingresos bajos |
| short_runtime + low_budget → low_rating | ~20% | >63% | >2.6 | Contenido barato y corto = peor calificación |

Estas reglas proveen **guías accionables** para la estrategia de contenido de CineVision Studios.


### 5.4 t-SNE — Validación Visual

La visualización t-SNE confirmó que:
- Los **3 clusters son coherentes** en el espacio de alta dimensión original.
- Existen **películas atípicas** (outliers) que merecen análisis individual.
- El eje de presupuesto crea **gradientes visuales claros**, alineados con la
  segmentación de K-Means.
- La estructura de datos es **no aleatoria y significativa** para la toma de
  decisiones.


### 5.5 Recomendaciones Estratégicas para CineVision Studios

Basadas en el análisis completo, se proponen las siguientes acciones:

**1. Estrategia de Diversificación de Portfolio:**
Mantener un mix de los 3 segmentos: 1–2 producciones blockbuster al año
(Cluster 2) para captura de revenue masivo, complementadas con 5–8 producciones
mainstream (Cluster 1) para flujo constante y 10–15 proyectos independientes
(Cluster 0) para construcción de marca artística y talento emergente.

**2. Coproducciones Internacionales para Escalar:**
Las reglas de asociación y el perfil del Cluster 2 revelan que las producciones
multi-país tienen mayores ingresos. CineVision debería establecer alianzas con
estudios en UK, Francia y Canadá para co-financiar proyectos de alto presupuesto
y acceder a créditos fiscales internacionales.

**3. Estrategia para Cine en Idiomas Locales:**
El Cluster 0 y las reglas de asociación muestran que el cine no anglófono de
bajo presupuesto raramente alcanza altos ingresos por distribución tradicional.
CineVision debe orientar estas producciones hacia **festivales internacionales**
(Sundance, Cannes, BAFICI) y plataformas de streaming para maximizar retorno
con inversión mínima.

**4. Inversión en Calidad de Elenco:**
PC2 (Calidad y Elenco) muestra que `castWomenAmount` y `actorsAmount` correlacionan
con `voteAvg`. Producir películas con elencos más diversos y numerosos mejora
la calificación de audiencias — un activo de largo plazo para la reputación del
estudio.

**5. Segmentación para Marketing y Distribución:**
Utilizar los 3 clusters como base para **estrategias de marketing diferenciadas**:
presupuesto publicitario masivo para Cluster 2, marketing digital segmentado para
Cluster 1, y marketing de comunidad/festivales para Cluster 0. Los componentes
PCA pueden alimentar un **modelo de predicción de ROI** que optimice la asignación
de recursos de marketing pre-estreno.
